In [1]:
import os
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

In [2]:
import tensorflow as tf
from pathlib import Path

2026-02-19 17:58:27.137478: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-19 17:58:27.220873: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-19 17:58:27.256295: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
from mobilenetv2ssd.core.config import load_config

In [4]:
experiment_path = "../configs/experiments/exp001_baseline.yaml"

In [5]:
config = load_config(experiment_path)

In [6]:
from datasets.voc import VOCDataset

In [7]:
data = VOCDataset(root = config['data']['root'], split = "train", classes_file = config['data']['classes_file'], use_difficult = False)

In [8]:
data[1].path

'/mnt/d/dev/MobileNetV2-SSD/datasets/VOCdevkit/VOC2012/JPEGImages/2008_000015.jpg'

In [9]:
def _bytes_features(value):
    # This function takes values from String and Byte types
    return tf.train.Feature(bytes_list= tf.train.BytesList(value=value))

In [10]:
def _float_features(value):
    # This function takes values from Float and Double Values
    return tf.train.Feature(float_list= tf.train.FloatList(value=value))

In [11]:
def _int64_features(value):
    # This function takes values from (bool, enum, int)
    return tf.train.Feature(int64_list= tf.train.Int64List(value=value))

In [12]:
def parse_boxes(boxes):
    # Parsing the boxes
    flattened_boxes = boxes.flatten().tolist()
    return _float_features(flattened_boxes), _int64_features([len(boxes)])

In [13]:
def parse_labels(labels):
    flattened_labels = labels.flatten().tolist()
    return _int64_features(flattened_labels)

In [14]:
def parse_image_size(height, width):
    if isinstance(height,int):
       height= [height]

    if isinstance(width, int):
        width= [width]

    return _int64_features(height), _int64_features(width)

In [15]:
def parse_image_id(image_id):
    if isinstance(image_id, str):
        image_id = [image_id.encode('utf-8')]
        
    return _bytes_features(image_id)

In [16]:
def parse_raw_image(jpeg_bytes):
    if isinstance(jpeg_bytes,bytes):
        jpeg_bytes = [jpeg_bytes]

    return _bytes_features(jpeg_bytes)

In [17]:
def encode_features(boxes: tf.train.Feature, boxes_count: tf.train.Feature, labels: tf.train.Feature, height: tf.train.Feature, width: tf.train.Feature, image_id: tf.train.Feature, image_bytes: tf.train.Feature):
    feature = {
        'image/encoded': image_bytes,
        'image/height': height,
        'image/width': width,
        'image/boxes': boxes,
        'image/boxes_count': boxes_count,
        'image/labels': labels,
        'image/image_id': image_id,
    }

    example = tf.train.Example(features = tf.train.Features(feature= feature))
    return example.SerializeToString()

In [18]:
def encode_sample(sample):
    # Encode an image sample
    
    boxes = sample.boxes
    labels = sample.labels
    H,W = sample.orig_size
    id_ = sample.image_id

    # Reading the JPEG file
    with open(sample.path,"rb") as file:
        jpeg_bytes = file.read()

    encoded_bytes = parse_raw_image(jpeg_bytes)
    encoded_boxes, encoded_boxes_count = parse_boxes(boxes)
    encoded_labels = parse_labels(labels)
    encoded_height, encoded_width = parse_image_size(H,W)
    encoded_image_id = parse_image_id(id_)

    # Encoding the features into image
    example = encode_features(boxes= encoded_boxes,boxes_count= encoded_boxes_count, labels= encoded_labels, height= encoded_height, width= encoded_width,image_id= encoded_image_id, image_bytes= encoded_bytes) 

    return example

In [19]:
def write_serialized_record(example, writer):
    if writer is None:
        raise ValueError("tf.io.TFRecordWriter is None")

    # Writing the example
    writer.write(example)

In [27]:
def write_split(dataset, output_dir: Path, split_name, shard_size= 500):
    shard_index= 0
    shard_img_count= 0

    output_dir= Path(output_dir)
    main_dir= output_dir / split_name
    main_dir.mkdir(parents= True, exist_ok= True)
    output_dir= output_dir / split_name / f"{shard_index:03d}.tfrecord"
    writer= tf.io.TFRecordWriter(str(output_dir))

    # Looping through the dataset
    for sample in dataset:
        # Creating the record
        record = encode_sample(sample)
        write_serialized_record(record, writer)
        shard_img_count= shard_img_count + 1

        if shard_img_count >= shard_size:
            # Creating a new shard
            writer.close()
            print(f"Finished Writing Shard: {shard_index:03d}")
            shard_index= shard_index + 1
            shard_img_count= 0
            shard_dir = main_dir / f"{shard_index:03d}.tfrecord"
            writer= tf.io.TFRecordWriter(str(shard_dir))

    writer.close()

In [28]:
write_split(data, output_dir= Path("./shards"), split_name= "train")

Finished Writing Shard: 000
Finished Writing Shard: 001
Finished Writing Shard: 002
Finished Writing Shard: 003


KeyboardInterrupt: 